## Descargar y preparar imágenes de personas desde Open Images V7

**Objetivo:** crear automáticamente un conjunto **inicial** de imágenes con recuadros de personas en Google Drive, para usarlo en el cuaderno `entrenar_personas.ipynb`. **No es el entrenamiento**: no requiere GPU y no descarga los videos de evaluación BEHAVE, KTH ni UT-Interaction.

Fuente: [Open Images V7 oficial](https://storage.googleapis.com/openimages/web/factsfigures_v7.html), descargado mediante [FiftyOne](https://docs.voxel51.com/dataset_zoo/datasets/open_images_v7.html). Imágenes **listadas** como CC BY 2.0; anotaciones CC BY 4.0. Google no garantiza las licencias individuales: **comprobar antes de redistribuir**. Aquí solo se guardan los datos en **tu Google Drive** y un manifiesto de procedencia; **no** se suben datos a GitHub.

**Nota de calidad:** Open Images puede tener anotaciones de personas incompletas, especialmente en el conjunto de entrenamiento. Revisa visualmente los recuadros (en particular multitudes/oclusiones). Con 260/60 imágenes solo prepararemos un **experimento pequeño**; no se garantiza una mejora sobre `yolo11n.pt`.

**Pasos:** ejecuta las celdas en orden, concede permiso a Drive cuando Colab lo solicite y revisa las muestras dibujadas al final.
**Corrección:** FiftyOne suele guardar las cajas de Open Images en `ground_truth`. El cuaderno identifica el campo existente para evitar errores `KeyError`.


In [ ]:
%pip -q install "fiftyone>=1.0,<2" "pyyaml>=6" "pillow>=10"

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/detector_cascos_motos/datasets/personas_v1')
for split in ('train', 'val'):
    for kind in ('images', 'labels'):
        (BASE / kind / split).mkdir(parents=True, exist_ok=True)
# Nunca borrar ni sobreescribir un dataset previo sin revisar.
ocupados = [str(p) for split in ('train', 'val') for kind in ('images', 'labels') for p in (BASE/kind/split).iterdir() if p.is_file()]
if ocupados:
    raise RuntimeError('El dataset ya tiene archivos. No se sobrescribirá. Revisa primero: ' + str(ocupados[:5]))
print('Directorio de destino vacío y listo:', BASE)

In [ ]:
import fiftyone.zoo as foz
# Descarga PARCIAL, no el Open Images completo (~561 GB).
# split=train y split=validation son conjuntos distintos.
LIMITES = {'train': 260, 'validation': 60}
conjuntos = {}
for original_split, cantidad in LIMITES.items():
    print('Descargando Open Images V7', original_split, 'Person:', cantidad, 'imágenes como máximo')
    conjuntos[original_split] = foz.load_zoo_dataset(
        'open-images-v7',
        split=original_split,
        label_types=['detections'],
        classes=['Person'],
        only_matching=True,
        max_samples=cantidad,
        shuffle=True,
        seed=42,
        num_workers=4,
        dataset_name=f'cascos_personas_oiv7_{original_split}_{cantidad}',
    )
    print('Recibidas:', len(conjuntos[original_split]))

In [ ]:
import csv
import shutil
from collections import Counter
import math

conteos = {}
filas = []
ids_por_split = {}
for fuente, destino_split in (('train','train'), ('validation','val')):
    conjunto = conjuntos[fuente]
    total = 0
    cajas_total = 0
    ids_originales = set()
    for sample in conjunto:
        # Open Images V7 usa habitualmente 'ground_truth', otras importaciones 'detections'.
        campos = sample.field_names
        campo_cajas = next((campo for campo in ('ground_truth', 'detections') if campo in campos), None)
        if campo_cajas is None:
            raise RuntimeError(f'No se encontró el campo de anotaciones de personas. Disponibles: {campos}')
        anotacion = sample[campo_cajas]
        detecciones = list(anotacion.detections) if anotacion is not None else []
        cajas = []
        for det in detecciones:
            if det.label != 'Person':
                continue
            x, y, w, h = map(float, det.bounding_box)  # formato FiftyOne: x-esquina, y-esquina, ancho, alto
            if not all(math.isfinite(t) for t in (x,y,w,h)) or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                raise ValueError('Caja no válida en Open Images: ' + str(sample.filepath))
            if x+w > 1.0001 or y+h > 1.0001:
                raise ValueError('Caja fuera del área de imagen: ' + str(sample.filepath))
            xc, yc = x + w/2, y + h/2
            cajas.append(f'0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')
        if not cajas:
            continue  # no inventar anotaciones ausentes
        origen = Path(sample.filepath)
        if not origen.is_file():
            raise FileNotFoundError(str(origen))
        original_id = origen.stem
        nombre_imagen = f'oi7_{destino_split}_{original_id}{origen.suffix.lower()}'
        imagen_destino = BASE/'images'/destino_split/nombre_imagen
        etiqueta_destino = BASE/'labels'/destino_split/(Path(nombre_imagen).stem + '.txt')
        if imagen_destino.exists() or etiqueta_destino.exists():
            raise FileExistsError('Hay un archivo repetido: ' + str(imagen_destino))
        shutil.copy2(origen, imagen_destino)
        etiqueta_destino.write_text('\n'.join(cajas)+'\n', encoding='utf-8')
        total += 1
        cajas_total += len(cajas)
        ids_originales.add(original_id)
        filas.append({'origen': 'Open Images V7', 'split_original': fuente, 'imagen_id': original_id, 'archivo': nombre_imagen, 'personas_etiquetadas': len(cajas)})
    if total < 10:
        raise ValueError(f'Hay muy pocas imágenes válidas en {destino_split}: {total}')
    conteos[destino_split] = {'imagenes': total, 'cajas_persona': cajas_total}
    ids_por_split[destino_split] = ids_originales
if ids_por_split['train'] & ids_por_split['val']:
    raise ValueError('Se encontraron imágenes originales comunes entre train y val. No continuar.')
MANIFEST = BASE/'procedencia_open_images_v7.csv'
with MANIFEST.open('w', encoding='utf-8', newline='') as archivo:
    writer = csv.DictWriter(archivo, fieldnames=['origen','split_original','imagen_id','archivo','personas_etiquetadas'])
    writer.writeheader()
    writer.writerows(filas)
(BASE/'LICENCIA_Y_REVISION.txt').write_text(
    'Fuente: https://storage.googleapis.com/openimages/web/factsfigures_v7.html\n'
    'Imágenes listadas como CC BY 2.0, anotaciones Google CC BY 4.0.\n'
    'Google no asegura el estado legal de las imágenes individuales. Verificar derechos y atribuciones antes de compartir.\n'
    'Los IDs en procedencia_open_images_v7.csv permiten identificar las imágenes de origen.\n'
    'No subir imágenes ni etiquetas a GitHub sin revisar permisos.\n'
    'Revisar que todas las personas estén anotadas, especialmente las superpuestas.\n', encoding='utf-8')
print('Exportación finalizada:', conteos)
print('Procedencia:', MANIFEST)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Rectangle
for split in ('train','val'):
    ejemplos = sorted((BASE/'images'/split).glob('*'))[:3]
    fig, axs = plt.subplots(1, len(ejemplos), figsize=(5*len(ejemplos),4))
    if len(ejemplos) == 1:
        axs = [axs]
    for ax, imgfile in zip(axs, ejemplos):
        img = Image.open(imgfile).convert('RGB')
        w0, h0 = img.size
        ax.imshow(img)
        etiqueta = BASE/'labels'/split/(imgfile.stem+'.txt')
        for l in etiqueta.read_text().splitlines():
            _, xc, yc, w, h = map(float, l.split())
            ax.add_patch(Rectangle(((xc-w/2)*w0, (yc-h/2)*h0), w*w0, h*h0, fill=False, edgecolor='lime', linewidth=2))
        ax.set_title(f'{split}: {imgfile.name[:20]}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
print('IMPORTANTE: inspecciona más imágenes en Drive y corrige las etiquetas incompletas antes de entrenar.')

### Después de descargar

Abre [entrenar_personas.ipynb](https://colab.research.google.com/github/eduardogrados644-lang/detector_cascos_motos/blob/mejoras-modulos-vial-conductual/entrenamientos_colab/entrenar_personas.ipynb) desde GitHub. En el cuaderno de entrenamiento, activa la GPU, ejecuta sus celdas y usa `best.pt` solo después de validar que funciona mejor que `yolo11n.pt` en videos reservados.

Este cuaderno **no** entrena modelos de acciones, armas o intención; solo prepara anotaciones de la clase `person`. Tampoco mejora por sí solo los IDs ni la pérdida de seguimiento; esos cambios serán una etapa de ingeniería separada.